In [ ]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import setting_path
from eval.persona_ablation.personas import CONDITIONS
from eval.persona_ablation.run import REPEATS

In [ ]:
eval_db = setting_path("SIMULATION_DB")

with sqlite3.connect(eval_db) as conn:
    runs = pd.read_sql_query(
        """
        SELECT
            eval_id,
            substr(name, length('eval_') + 1) AS name,
            model_name,
            persona_limit,
            status,
            input_tokens,
            output_tokens
        FROM eval_runs
        WHERE name LIKE 'eval_%'
        ORDER BY eval_id
        """,
        conn,
    )
    predictions = pd.read_sql_query(
        """
        SELECT
            substr(r.name, length('eval_') + 1) AS name,
            r.eval_id,
            p.persona_id,
            p.item_id,
            p.predicted,
            p.probabilities,
            p.actual_label,
            p.correct
        FROM eval_wave4_predictions AS p
        JOIN eval_runs AS r ON r.eval_id = p.eval_id
        WHERE r.status = 'success' AND r.name LIKE 'eval_%'
        """,
        conn,
    )

with sqlite3.connect(setting_path("PERSONA_DB")) as conn:
    questions = pd.read_sql_query(
        """
        SELECT item_id, question_type, options_json
        FROM questions
        WHERE LOWER(question_type) IN ('mc', 'matrix')
          AND is_overlap = 1
        """,
        conn,
    )

questions["options"] = questions["options_json"].map(json.loads)
predictions = predictions.merge(
    questions[["item_id", "question_type", "options"]],
    on="item_id",
)


def scaled_accuracy(row: pd.Series) -> float:
    """Return 1 minus the absolute error divided by the option span."""
    options = row["options"]
    human = options.index(row["actual_label"])
    if str(row["question_type"]).lower() == "matrix":
        probabilities = json.loads(row["probabilities"])
        ai = sum(int(level) * weight for level, weight in probabilities.items())
    else:
        ai = options.index(row["predicted"])
    return 1 - abs(human - ai) / (len(options) - 1)


predictions["accuracy"] = predictions.apply(scaled_accuracy, axis=1)
runs

# Ablation

Accuracy is `1 - abs(human - ai) / (number of options - 1)`. A multiple-choice answer uses the index of the chosen option. A matrix answer uses the probability-weighted score, and the human answer uses the index of their label. The span is the distance from the first option to the last. Cost uses Jev's published rate of $0.042 per million input tokens. Output tokens are free. Tokens are counted once per model call, so a batched question is not charged once per sub-item.

In [ ]:
INPUT_USD_PER_MILLION = 0.042

ablation = predictions.loc[predictions["name"].isin(CONDITIONS)].copy()
ablation["name"] = pd.Categorical(ablation["name"], CONDITIONS, ordered=True)
accuracy = ablation.groupby("name", observed=True).agg(
    answers=("accuracy", "size"),
    accuracy=("accuracy", "mean"),
)

usage = (
    runs.loc[runs["name"].isin(CONDITIONS) & (runs["status"] == "success")]
    .groupby("name", observed=False)[["input_tokens", "output_tokens"]]
    .sum()
)
summary = accuracy.join(usage)
summary["cost_usd"] = summary["input_tokens"] * INPUT_USD_PER_MILLION / 1_000_000
summary

# Repeatability

`persona_summary` is the first full-summary pass. `repeatability-2` through `repeatability-5` are the other four.

Accuracy uses the same option-scale score as the ablation. Same option is the share of questions where all five passes chose the same label, whether or not that label is correct.

In [ ]:
repeat_names = ["persona_summary", *[f"repeatability-{repeat}" for repeat in range(2, REPEATS + 1)]]
repeats = predictions.loc[predictions["name"].isin(repeat_names)].copy()
repeats["name"] = pd.Categorical(repeats["name"], repeat_names, ordered=True)

by_answer = repeats.groupby(["persona_id", "item_id"]).agg(
    passes=("predicted", "size"),
    distinct_predictions=("predicted", "nunique"),
)
complete = by_answer.loc[by_answer["passes"] == len(repeat_names)]

pd.Series(
    {
        "accuracy": repeats["accuracy"].mean(),
        "same_option": (complete["distinct_predictions"] == 1).mean(),
    }
)

# Full persona

This section uses the successful `persona_summary` run: the full stored summary for the first 100 personas, scored on the wave 4 multiple-choice and matrix questions each person answered. Accuracy is the option-scale score from the cells above. A prediction counts as wrong when the chosen label differs from the wave 4 label.

The same-answer check also uses the four extra full-summary passes, `repeatability-2` through `repeatability-5`.

The questions below ask where that full summary fails, whether the failure is shared across demographic groups, and whether the extra detail beats a simpler baseline.


In [ ]:
import matplotlib.pyplot as plt

full = predictions.loc[predictions["name"] == "persona_summary"].copy()
full["wrong"] = full["predicted"] != full["actual_label"]

with sqlite3.connect(setting_path("PERSONA_DB")) as conn:
    catalog = pd.read_sql_query(
        """
        SELECT item_id, question_id, category, question_text, item_text
        FROM questions
        """,
        conn,
    )
    people = pd.read_sql_query(
        """
        SELECT persona_id, gender, region, age_band, income_band, education_level
        FROM personas
        """,
        conn,
    )

catalog["category"] = catalog["category"].str.strip()
full = full.merge(catalog, on="item_id").merge(people, on="persona_id")
full["region"] = full["region"].map(
    {
        "West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)": "West",
        "South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)": "South",
        "Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)": "Northeast",
        "Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)": "Midwest",
    }
)


def option_distance(row: pd.Series) -> float:
    """Return how far the model answer sits from the human answer, in option steps."""
    options = row["options"]
    human = options.index(row["actual_label"])
    if str(row["question_type"]).lower() == "matrix":
        probabilities = json.loads(row["probabilities"])
        ai = sum(int(level) * weight for level, weight in probabilities.items())
    else:
        ai = options.index(row["predicted"])
    return abs(human - ai)


full["distance"] = full.apply(option_distance, axis=1)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "font.size": 11,
    }
)
BAR = "#2f5d8c"


## Which questions are wrong most often

Each bar is one `question_id`. The length is the share of answers whose label does not match the person. The number at the end of the bar is the option-scale accuracy, so a near miss on a long scale stays visible.


In [ ]:
questions = full.groupby(["question_id", "category"], as_index=False).agg(
    answers=("wrong", "size"),
    wrong=("wrong", "mean"),
    accuracy=("accuracy", "mean"),
)
questions["label"] = questions["question_id"] + " · " + questions["category"]
worst = questions.sort_values("wrong", ascending=False).head(12).sort_values("wrong")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(worst["label"], worst["wrong"], color=BAR)
for top, wrong, accuracy in zip(worst["label"], worst["wrong"], worst["accuracy"]):
    ax.text(wrong + 0.01, top, f"{accuracy:.2f}", va="center", fontsize=9)
ax.set_xlim(0, 1.12)
ax.set_xlabel("Share of answers with the wrong label")
ax.set_title("Questions the full persona misses most often")
fig.tight_layout()
plt.show()

questions.sort_values("wrong", ascending=False)[
    ["question_id", "category", "answers", "wrong", "accuracy"]
].head(12)


The disease-loss question (`QID158`) is wrong for 96% of the people who answered it. In that loss frame the model only chooses program A, while the wave 4 answers lean toward program B. The gain frame (`QID157`) has the same shape: the model piles onto program A and misses 81% of people.

The jacket question is the farthest miss. It is a yes/no, so a wrong label scores 0, and its option-scale accuracy is 0.30. Three quarters of these people would not make the extra trip for the cheaper jacket. The model says yes for 88% of them.

`Anchoring - redwood low` is perfect because every person, and the model, said the tallest redwood is more than 85 feet. That is a shared obvious answer, so it does not show that the summary is tracking the person. The African-countries anchor with a high number, and both probability-matching problems, are the other questions the model gets right relatively often.


## Which categories are wrong most often

The category is the question's stored `category`, which is finer than multiple choice versus matrix. Many categories are a single question, so this ranking is close to the question ranking. Product preferences are the exception: 40 pricing questions and about half of all scored answers.


In [ ]:
categories = full.groupby("category", as_index=False).agg(
    answers=("wrong", "size"),
    questions=("question_id", "nunique"),
    wrong=("wrong", "mean"),
    accuracy=("accuracy", "mean"),
)
ranked = categories.sort_values("wrong")

fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(ranked["category"], ranked["wrong"], color=BAR)
ax.set_xlim(0, 1)
ax.set_xlabel("Share of answers with the wrong label")
ax.set_title("Wrong-label rate by question category")
fig.tight_layout()
plt.show()

categories.sort_values(["wrong", "accuracy"], ascending=[False, True])


Disease-loss, the willingness-to-pay certainty question, and the disease-gain frame are wrong most often. Their option-scale scores are higher than the jacket question and Allais Form 2, because those two are short scales: a miss lands at the other end.

Product preferences, which are half the answers, are wrong about half the time and score 0.51. They pull the overall accuracy down to 0.63. Linda, false consensus, and the heuristics-and-biases items are wrong often, but the miss is usually a neighboring point on the scale, so their option-scale scores stay near 0.74 to 0.80.

Anchoring is the strongest category family. The low redwood anchor is exact. The high African-countries anchor is wrong only 13% of the time.


## How often the same answer is chosen

For each person and question, all five full-summary passes chose a label. This is the share of those pairs where all five labels match. It does not ask whether the shared label is the person's wave 4 answer.


In [ ]:
repeat_rows = predictions.loc[
    predictions["name"].isin(repeat_names), ["persona_id", "item_id", "predicted"]
]
agreement = repeat_rows.groupby(["persona_id", "item_id"]).agg(
    passes=("predicted", "size"),
    distinct_predictions=("predicted", "nunique"),
)
agreement = agreement.loc[agreement["passes"] == len(repeat_names)].reset_index()
agreement["same"] = agreement["distinct_predictions"] == 1
agreement = agreement.merge(catalog[["item_id", "category"]], on="item_id")

same_by_category = (
    agreement.groupby("category")
    .agg(questions=("same", "size"), same_answer=("same", "mean"))
    .reset_index()
    .sort_values("same_answer")
)
unstable = same_by_category.loc[same_by_category["same_answer"] < 0.9].sort_values("same_answer")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(unstable["category"], unstable["same_answer"], color=BAR)
ax.set_xlim(0, 1)
ax.axvline(agreement["same"].mean(), color="#c47b2b", linestyle="--", label="All questions")
ax.set_xlabel("Share of person-questions with the same label on all five passes")
ax.set_title("Categories where the full persona changes its answer")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

pd.Series(
    {
        "person_questions": len(agreement),
        "same_answer": agreement["same"].mean(),
    }
)


The five passes agree on 91.8% of person-questions. The label is stable even when it is the wrong one.

The unstable categories are the willingness-to-accept certainty question (46% agreement), the two weaker "less is more" gambles, both outcome-bias stories, and the two disease frames. Anchoring and both probability-matching problems agree on every pass.


## Accuracy across demographics

Each bar is the option-scale accuracy for people in that group. The axis runs from 0 to 1 so a gap of a few hundredths stays in proportion. The dashed line is the accuracy of the full persona on everyone.


In [ ]:
overall_accuracy = full["accuracy"].mean()
groups = {
    "Gender": (["Female", "Male"], "gender"),
    "Region": (["Northeast", "Midwest", "South", "West"], "region"),
    "Age": (["18-29", "30-49", "50-64", "65+"], "age_band"),
    "Income": (
        [
            "Less than $30,000",
            "$30,000-$50,000",
            "$50,000-$75,000",
            "$75,000-$100,000",
            "$100,000 or more",
        ],
        "income_band",
    ),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (title, (order, column)) in zip(axes.ravel(), groups.items()):
    scores = (
        full.groupby(column)["accuracy"].mean().reindex(order)
    )
    ax.barh(scores.index.astype(str), scores.values, color=BAR)
    ax.axvline(overall_accuracy, color="#c47b2b", linestyle="--")
    ax.set_xlim(0, 1)
    ax.set_title(title)
    for label, value in scores.items():
        ax.text(value + 0.02, str(label), f"{value:.3f}", va="center", fontsize=8)
fig.suptitle("Full-persona accuracy by demographic group", y=1.02)
fig.tight_layout()
plt.show()

education = full.groupby("education_level").agg(
    people=("persona_id", "nunique"),
    accuracy=("accuracy", "mean"),
)
education.sort_values("accuracy")


## What each group gets wrong

The bars compare women and men on the categories that are wrong most often and that both groups answered at least 20 times. Other demographic cuts are printed against the same categories.


In [ ]:
def category_wrong(column: str, order: list[str]) -> pd.DataFrame:
    """Return the wrong-label rate for each category and demographic group."""
    table = full.pivot_table(
        index="category",
        columns=column,
        values="wrong",
        aggfunc="mean",
    )
    return table.reindex(columns=order)


gender_wrong = category_wrong("gender", ["Female", "Male"])
eligible = gender_wrong.dropna()
counts = full.groupby(["category", "gender"]).size().unstack()
eligible = eligible.loc[(counts["Female"] >= 20) & (counts["Male"] >= 20)]
focus = eligible.assign(worst=eligible.max(axis=1)).sort_values("worst", ascending=False).head(6)
focus = focus.drop(columns="worst").sort_values("Female")

fig, ax = plt.subplots(figsize=(9, 4.5))
positions = range(len(focus))
ax.barh([position - 0.18 for position in positions], focus["Female"], height=0.36, color="#2f5d8c", label="Female")
ax.barh([position + 0.18 for position in positions], focus["Male"], height=0.36, color="#c47b2b", label="Male")
ax.set_yticks(list(positions), focus.index)
ax.set_xlim(0, 1)
ax.set_xlabel("Share of answers with the wrong label")
ax.set_title("Weak categories for women and men")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

from IPython.display import display

display(focus.round(3))

income_order = [
    "Less than $30,000",
    "$30,000-$50,000",
    "$50,000-$75,000",
    "$75,000-$100,000",
    "$100,000 or more",
]
pricing = full.loc[full["category"] == "Product Preferences - Pricing"]
pricing.groupby("income_band")["wrong"].mean().reindex(income_order)


Demographic groups sit close together. Women score 0.622 and men 0.631. The four regions span 0.614 in the Midwest to 0.634 in the West. Age bands span 0.622 to 0.629.

Income moves a little more. People reporting $30,000 to $50,000 score 0.600, and people reporting $50,000 to $75,000 score 0.642. That gap shows up on the pricing questions, which are half the answers: the lower band is wrong on 53% of purchases and the higher band on 48%. It does not come from one band missing a different psychological category.

High-school graduates are the weakest education group, at 0.595. One person with less than a high-school education is too small a group to read.

Women and men miss the same hard categories. Disease-loss is wrong for about 96% of each, and the jacket question, outcome-bias story, and Linda items stay high for both. Myside Ford is the exception: women are wrong on 72% of 36 answers, and men on 48% of 27. Disease-gain is also higher for women (88% of 26) than for men (71% of 21). Those cells are small next to the 4,000 pricing answers, which is why the overall scores still sit within a point of each other. Region and age do not develop a different list of weak categories.


## Three checks the mission asks for

Individual accuracy can look acceptable while the model still ignores the person. These checks ask three follow-up questions.

- When the label is wrong, how far is it from the person's answer?
- Does the mix of predicted answers match the mix of wave 4 answers?
- Does the full summary beat the most common wave 4 answer, and the demographics-only persona, by enough to justify its cost?

The most common answer is the label chosen by the largest share of these 100 people on that question. It uses their wave 4 answers, so it is a ceiling for "ignore the person and guess the crowd," not a prediction made in advance.


In [ ]:
miss = pd.Series("more than one step", index=full.index)
miss = miss.mask(full["distance"] <= 1, "within one step")
miss = miss.mask(~full["wrong"], "exact")
miss_share = miss.value_counts(normalize=True).reindex(
    ["exact", "within one step", "more than one step"]
)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(miss_share.index, miss_share.values, color=BAR)
ax.set_xlim(0, 1)
ax.set_xlabel("Share of full-persona answers")
ax.set_title("How far the answer is from the person")
for label, value in miss_share.items():
    ax.text(value + 0.02, label, f"{value:.0%}", va="center")
fig.tight_layout()
plt.show()


def answer_mix(question_id: str) -> pd.DataFrame:
    """Return the human and model share of each option, in option order."""
    rows = full.loc[full["question_id"] == question_id]
    options = rows["options"].iloc[0]
    return pd.DataFrame(
        {
            "Wave 4": rows["actual_label"].value_counts(normalize=True).reindex(options, fill_value=0),
            "Full persona": rows["predicted"].value_counts(normalize=True).reindex(options, fill_value=0),
        }
    )


fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, question_id, title in zip(
    axes,
    ["QID158", "QID184"],
    ["Disease-loss", "Jacket"],
):
    mix = answer_mix(question_id)
    positions = range(len(mix))
    ax.barh([p - 0.18 for p in positions], mix["Wave 4"], height=0.36, color="#2f5d8c", label="Wave 4")
    ax.barh([p + 0.18 for p in positions], mix["Full persona"], height=0.36, color="#c47b2b", label="Full persona")
    ax.set_yticks(list(positions), mix.index)
    ax.set_xlim(0, 1)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("The predicted mix against the wave 4 mix", y=1.02)
fig.tight_layout()
plt.show()

mode = full.groupby("item_id")["actual_label"].agg(lambda labels: labels.value_counts().index[0])
crowd = full[["persona_id", "item_id"]].copy()
crowd["predicted"] = crowd["item_id"].map(mode)
crowd["actual_label"] = full["actual_label"].to_numpy()
crowd["options"] = full["options"].to_numpy()
crowd["accuracy"] = [
    1 - abs(options.index(actual) - options.index(guess)) / (len(options) - 1)
    for options, actual, guess in zip(crowd["options"], crowd["actual_label"], crowd["predicted"])
]
demographic = predictions.loc[predictions["name"] == "demographic", "accuracy"].mean()
comparison = pd.Series(
    {
        "Most common wave 4 answer": crowd["accuracy"].mean(),
        "Demographics only": demographic,
        "Full persona": full["accuracy"].mean(),
    }
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(comparison.index, comparison.values, color=BAR)
ax.set_xlim(0, 1)
ax.set_xlabel("Option-scale accuracy")
ax.set_title("Full persona against two baselines")
for label, value in comparison.items():
    ax.text(value + 0.02, label, f"{value:.3f}", va="center")
fig.tight_layout()
plt.show()

full_cost = runs.loc[
    (runs["name"] == "persona_summary") & (runs["status"] == "success"), "input_tokens"
].sum() * INPUT_USD_PER_MILLION / 1_000_000
demographic_cost = runs.loc[
    (runs["name"] == "demographic") & (runs["status"] == "success"), "input_tokens"
].sum() * INPUT_USD_PER_MILLION / 1_000_000
pd.Series(
    {
        "full_persona_accuracy": comparison["Full persona"],
        "demographic_accuracy": comparison["Demographics only"],
        "most_common_accuracy": comparison["Most common wave 4 answer"],
        "full_persona_cost_usd": full_cost,
        "demographic_cost_usd": demographic_cost,
    }
)


48% of answers match the label exactly, and another 38% land within one step on the option list. 15% are farther away. Most misses are close, which is why the option-scale score (0.626) is higher than the exact-match rate.

The mix of answers is a different failure. Across questions, the model's most common label takes 79% of people, while the most common wave 4 label takes 55%. On disease-loss the model never selects program B. On the jacket question it says yes for most people and the participants say no. The summary is producing a crowd of its own, rather than the crowd in wave 4.

The full summary does not beat guessing the most common wave 4 answer (0.668). It barely beats the demographics-only persona (0.620 versus 0.626). One full-summary pass costs about $0.90 in input tokens, against about $0.16 for demographics. The place the full text clearly helps is Allais Form 1, where accuracy rises from 0.39 with demographics alone to 0.71. On disease-loss, the willingness-to-accept certainty question, and the weaker "less is more" gamble, the full summary is worse than demographics.

Taken together, the full persona is stable, and gender, region, age, and income differ by only a few points. The weak categories are shared. Myside Ford is the one gender split large enough to notice, on 63 answers. The cost of the full summary does not buy a matching gain in accuracy. The misses that matter are the framing, pricing, and willingness-to-pay questions, where the model commits to one side and the participants do not.


## Spread of answers versus spread of predictions

Each point is one question item. The horizontal axis is the standard deviation of the wave 4 option index, and the vertical axis is the same measure for the label the full persona chose. The dashed line is where the two spreads match. A point below the line means the predictions vary less across people than the real answers do.

In [ ]:
full["human_index"] = [
    options.index(label) for options, label in zip(full["options"], full["actual_label"])
]
full["model_index"] = [
    options.index(label) for options, label in zip(full["options"], full["predicted"])
]
spread = full.groupby("item_id", as_index=False).agg(
    question_id=("question_id", "first"),
    question_type=("question_type", "first"),
    human_sd=("human_index", lambda values: values.std(ddof=0)),
    model_sd=("model_index", lambda values: values.std(ddof=0)),
)

fig, ax = plt.subplots(figsize=(7, 7))
for question_type, color in (("MC", "#2f5d8c"), ("Matrix", "#c47b2b")):
    part = spread.loc[spread["question_type"].str.lower() == question_type.lower()]
    ax.scatter(part["human_sd"], part["model_sd"], s=28, color=color, label=question_type, alpha=0.85)
limit = max(spread["human_sd"].max(), spread["model_sd"].max()) * 1.08
ax.plot([0, limit], [0, limit], linestyle="--", color="#888888", linewidth=1)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Spread of wave 4 answers, in option steps")
ax.set_ylabel("Spread of predictions, in option steps")
ax.set_title("Does the full persona vary as much as people do?")
ax.legend(frameon=False)

spread["gap"] = spread["human_sd"] - spread["model_sd"]
labels = pd.concat(
    [
        spread.nlargest(8, "gap").drop_duplicates("question_id"),
        spread.nsmallest(2, "gap").drop_duplicates("question_id"),
    ]
).drop_duplicates("question_id")
for row in labels.itertuples():
    ax.annotate(row.question_id, (row.human_sd, row.model_sd), fontsize=8, xytext=(5, 4), textcoords="offset points")
fig.tight_layout()
plt.show()

pd.Series(
    {
        "items": len(spread),
        "predictions_tighter": (spread["model_sd"] < spread["human_sd"]).mean(),
        "mean_human_sd": spread["human_sd"].mean(),
        "mean_model_sd": spread["model_sd"].mean(),
        "same_prediction_for_everyone": int((spread["model_sd"] == 0).sum()),
    }
)

Predictions are tighter than the wave 4 answers on 90% of items. The average spread is 0.80 option steps for people and 0.47 for the model. On 31 items the model gives every person the same label, so those points sit on the bottom axis.

Proportion dominance (`QID177` through `QID179`) and the heuristics-and-biases items (`QID288`, `QID289`) are the clearest collapses: people are spread across more than one option step, and the predictions barely move. `QID190`, the willingness-to-accept certainty question, is the exception in the other direction. There the model is more scattered than the participants.

In [ ]:
constant_rows = full.loc[full["item_id"].isin(spread.loc[spread["model_sd"] == 0, "item_id"])].copy()
constant_rows["agreed"] = constant_rows["actual_label"] == constant_rows["predicted"]
same_answer = (
    constant_rows.groupby("item_id", as_index=False)
    .agg(
        question_id=("question_id", "first"),
        category=("category", "first"),
        question=("question_text", "first"),
        item=("item_text", "first"),
        locked_answer=("predicted", "first"),
        people=("persona_id", "size"),
        people_who_agree=("agreed", "mean"),
    )
    .sort_values(["category", "question_id", "item_id"])
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
same_answer